In [1]:
#baitaptrenlopakt
import heapq
import copy

# Cấu hình cho bài toán 15-puzzle
N = 4

class PuzzleNode:
    def __init__(self, board, empty_pos, g_cost, h_cost, parent=None):
        self.board = board
        self.empty_pos = empty_pos  # (row, col)
        self.g_cost = g_cost        # Chi phí từ gốc đến hiện tại (level)
        self.h_cost = h_cost        # Chi phí dự đoán đến đích (heuristic)
        self.f_cost = g_cost + h_cost
        self.parent = parent

    # Hỗ trợ so sánh trong Priority Queue (dựa trên f_cost)
    def __lt__(self, other):
        return self.f_cost < other.f_cost

    # Chuyển ma trận thành tuple để lưu vào set (visited)
    def get_state(self):
        return tuple(tuple(row) for row in self.board)

def calculate_manhattan_distance(current_board, goal_board):
    distance = 0
    # Tạo bản đồ vị trí đích để tra cứu nhanh
    goal_map = {}
    for r in range(N):
        for c in range(N):
            goal_map[goal_board[r][c]] = (r, c)

    for r in range(N):
        for c in range(N):
            val = current_board[r][c]
            if val != 0:  # Không tính ô trống
                target_r, target_c = goal_map[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance

def get_neighbors(node, goal_board):
    neighbors = []
    r, c = node.empty_pos
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)] # Lên, Xuống, Trái, Phải

    for dr, dc in directions:
        nr, nc = r + dr, c + dc
        if 0 <= nr < N and 0 <= nc < N:
            # Sao chép và hoán đổi vị trí ô trống
            new_board = [list(row) for row in node.board]
            new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]

            h_cost = calculate_manhattan_distance(new_board, goal_board)
            neighbors.append(PuzzleNode(new_board, (nr, nc), node.g_cost + 1, h_cost, node))

    return neighbors

def print_solution(node):
    path = []
    curr = node
    while curr:
        path.append(curr.board)
        curr = curr.parent

    print(f"Đã tìm thấy lời giải sau {len(path)-1} bước di chuyển ")
    for step, board in enumerate(reversed(path)):
        print(f"Bước {step}:")
        for row in board:
            print(" ".join(f"{val:2}" for val in row))

def solve_15_puzzle(initial_board, goal_board):
    # Tìm vị trí ô trống ban đầu
    start_empty = [(r, c) for r in range(N) for c in range(N) if initial_board[r][c] == 0][0]

    start_h = calculate_manhattan_distance(initial_board, goal_board)
    root = PuzzleNode(initial_board, start_empty, 0, start_h)

    open_list = []
    heapq.heappush(open_list, root)

    closed_list = set()
    closed_list.add(root.get_state())

    while open_list:
        current_node = heapq.heappop(open_list)

        if current_node.h_cost == 0:
            print_solution(current_node)
            return

        for neighbor in get_neighbors(current_node, goal_board):
            state = neighbor.get_state()
            if state not in closed_list:
                heapq.heappush(open_list, neighbor)
                closed_list.add(state)

    print("Không tìm thấy lời giải.") # Corrected placement

if __name__ == "__main__":
    # 0 đại diện cho ô trống
    start_state = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 0, 11],
        [13, 14, 15, 12]
    ]

    target_state = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 0]
    ]

    solve_15_puzzle(start_state, target_state)

Đã tìm thấy lời giải sau 2 bước di chuyển 
Bước 0:
 1  2  3  4
 5  6  7  8
 9 10  0 11
13 14 15 12
Bước 1:
 1  2  3  4
 5  6  7  8
 9 10 11  0
13 14 15 12
Bước 2:
 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14 15  0
